In [3]:
# %%
import glob
from pathlib import Path
 
import contextily as cx
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
from shapely.geometry import LineString, Point, shape
 
# %%
! ls ../data/processed/metadata/EFD_ULF | head
 
# %%
 
df = pd.read_csv(
    "/Users/emilyzhao/sample-project/CSES_data/CSES_01_EFD_1_L02_A1_213330_20211206_164953_20211206_172707_000.h5"
)
df
 
# %%
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Lon, df.Lat), crs="EPSG:4326")
gdf
 
 
# %%
 
ax = gdf.plot()
gdf.iloc[::10].plot(ax=ax, color="red", marker=".",)
 
# %%
# convert a list of points to a LineString
LineString(gdf.geometry)
 
# %%
ls = LineString(gdf.geometry)
ls.simplify(1)
 
 
# %%
 
from parsing import parse_filename
from tqdm import tqdm
 
# %%
# def get_semiorbit_id(fname):
#     if "EFD" in fname:
#         return Path(fname).stem.split("_")[6]
#     raise ValueError("Not an supported payload")
 
# %%
base_folder = Path("../data/processed/metadata/")
 
# %%
payload = "EFD_ULF"
 
# %%
payload_files = sorted(glob.glob(str(base_folder / payload / "*.csv")))
len(payload_files), payload_files[:5]
 
 
# %%
 
from datetime import datetime
import os
def extract_dates(file_name):
    try:
        base_name = os.path.basename(file_name) #returns the final component of a pathname
        parts = base_name.split('_')
        
        #find the index of the part that contains the start_date
        start_index = None
        for i in range(len(parts)):
            if parts[i].isdigit() and len(parts[i]) == 8:  # find the part with data format YYYYMMDD
                start_index = i
                break
        
        if start_index is None:
            raise ValueError(f"Formato data non trovato nel nome del file: {file_name}")
        
        start_date_str = '_'.join(parts[start_index:start_index + 2])
        end_date_str = '_'.join(parts[start_index + 2:start_index + 4])  
        
        start_date = datetime.strptime(start_date_str, '%Y%m%d_%H%M%S')
        end_date = datetime.strptime(end_date_str, '%Y%m%d_%H%M%S')
        
        return start_date, end_date
    except ValueError as e:
        print(f"Errore nel parsing delle date per il file {file_name}: {e}")
        return None, None
 
payload_metadata = []
 
for file in tqdm(payload_files):
    fn_data = parse_filename(Path(file).stem)
    df = pd.read_csv(file)
 
    start_date, end_date = extract_dates(file)
    
 
    df = df.dropna(subset=["Lat", "Lon"])
    if df.empty:
        continue
 
    # df = pd.read_csv(file, parse_dates=["UTCTime"])
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Lon, df.Lat), crs="EPSG:4326")
 
    # gdf["semiorbit_nr"] = fn_data["semiorbit_nr"]
    # gdf = gdf.drop(columns=["Lat", "Lon"])
 
    ls = LineString(gdf.geometry)
    ls = ls.simplify(.1)
    d = {
        "semiorbit_nr": fn_data["semiorbit_nr"],
        "geometry": ls,
        "data_start_time": gdf.UTCTime.min(),
        "data_end_time": gdf.UTCTime.max(),
        "name_start_time": start_date,
        "name_end_time": end_date,
    }
 
    payload_metadata += [d]
 
 
 
 
 
# %%
# payload_metadata = pd.concat(payload_metadata)
payload_metadata = gpd.GeoDataFrame(payload_metadata)
payload_metadata
 
 
 
# %%
# %%
# TODO Merge all the payload metadata into a single geodataframe removing all the duplcaite based on the semiorbit_nr and time_range. Keep the one with the longest time_range.
# Function to calculate time range in seconds
def calculate_time_range(row):
    return (row['data_end_time'] - row['data_start_time']).total_seconds()
 
 
file_paths = glob.glob(
    "../../../grp2/scientific-dashboard/data/processed/metadata/EFD_ULF/*csv"
)
 
 
 
 
# %%
payload_metadata = []
 
# Iterate through each CSV file
for file_path in file_paths:
    try:
        df = pd.read_csv(file_path)
        # Drop rows with missing Lat/Lon values
        df = df.dropna(subset=["Lat", "Lon"])
        if df.empty:
            continue
        # Extract semiorbit_nr from filename or any other relevant identifier
        file_name = Path(file_path).stem
        semiorbit_nr = file_name.split("_")[6]
        
        # Extract dates from filename or any other source
        start_date, end_date = extract_dates(file_path)
        
        # Compute geometry if applicable (assuming point data)
        geometry = gpd.points_from_xy(df.Lon, df.Lat)
 
        # Create GeoDataFrame
        gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
        
        # Calculate time range
        gdf['data_start_time'] = pd.to_datetime(gdf['data_start_time'])
        gdf['data_end_time'] = pd.to_datetime(gdf['data_end_time'])
        time_range = (gdf['data_end_time'] - gdf['data_start_time']).max().total_seconds()
 
        # Create a dictionary of metadata for this file
        metadata = {
            "semiorbit_nr": semiorbit_nr,
            "geometry": gdf.unary_union,  # Example: union of all geometries in gdf
            "data_start_time": gdf['data_start_time'].min(),
            "data_end_time": gdf['data_end_time'].max(),
            "time_range": time_range,
        }
        
        payload_metadata.append(metadata)
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
 
 
# %%
# Create a GeoDataFrame from the list of dictionaries
payload_gdf = gpd.GeoDataFrame(payload_metadata, geometry='geometry', crs="EPSG:4326")
 
# %%
# Sort by 'time_range' descending
payload_gdf = payload_gdf.sort_values(by='time_range', ascending=False)
 
# Drop duplicates based on 'semiorbit_nr', keeping the first (longest time_range)
payload_gdf = payload_gdf.drop_duplicates(subset='semiorbit_nr', keep='first')
 
# Optionally drop the 'time_range' column if not needed further
payload_gdf = payload_gdf.drop(columns=['time_range'])
 
payload_gdf
 
# %%
 
 
# create dataframe of semiorbits and fname for all payloads
# long format + compatto
[
    {
        "semiorbit_nr": "027321",
        "payload": "EFD_ULF",
        "fname": "CSES_01_EFD_1_L02_A1_027321_20180731_233412_20180801_001019_000.csv",
    }
]
 
# wide format
[
    {
        "semiorbit_nr": "027321",
        "EFD_ULF": "CSES_01_EFD_1_L02_A1_027321_20180731_233412_20180801_001019_000.csv",
        "EFD_HF": "CSES_01_EFD_1_L02_A1_027321_20180731_233412_20180801_001019_000.csv",
        # ...
    }
]
 
 
 
# %%
# fig, ax = plt.subplots(figsize=(10, 10))
 
# payload_metadata.iloc[:10000].to_crs("epsg:3857").plot(alpha=0.1 ,marker=".", ax=ax)
# cx.add_basemap(ax)
 
 
payload_metadata
 
 
fig, ax = plt.subplots(figsize=(10, 10))
 
# payload_metadata.iloc[:2].set_crs("epsg:4326").to_crs("epsg:3857").plot(alpha=1, ax=ax)
payload_metadata.iloc[:2].plot(alpha=1, ax=ax)
cx.add_basemap(ax)
 
 
 
version = "v3"
payload_metadata.iloc[:].to_file(f"./data/processed/{payload}_metadata_{version}.gpkg")
 
 
payload_metadata
 
# %%
 
 

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte